In [6]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

In [7]:
with open("../data/bhagavad_gita_sft_no_sanskrit.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} conversations")

Loaded 620 conversations


In [8]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge(ids, pair, new_id):
    new_ids = []
    i = 0

    while i < len(ids):
        if (
            i + 1 < len(ids)
            and ids[i] == pair[0]
            and ids[i + 1] == pair[1]
        ):
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


# Build token ID -> byte sequence mapping.
def build_vocab(merges):
    vocab = {
        token_id: bytes([token_id])
        for token_id in range(256)
    }

    for pair, new_id in merges.items():
        vocab[new_id] = (
            vocab[pair[0]] + vocab[pair[1]]
        )

    return vocab


def encode(text, merges):
    ids = list(str(text).encode("utf-8"))

    # Earlier learned merges have higher priority.
    merge_ranks = {
        pair: rank
        for rank, pair in enumerate(merges.keys())
    }

    while len(ids) >= 2:
        stats = get_stats(ids)

        # Find pairs that exist in the learned merges.
        valid_pairs = [
            pair
            for pair in stats
            if pair in merges
        ]

        if not valid_pairs:
            break

        # Select the earliest learned pair.
        best_pair = min(
            valid_pairs,
            key=lambda pair: merge_ranks[pair],
        )

        ids = merge(
            ids,
            best_pair,
            merges[best_pair],
        )

    return ids


def decode(ids, vocab):
    byte_sequence = b"".join(
        vocab[token_id]
        for token_id in ids
    )

    return byte_sequence.decode("utf-8")

In [9]:
def load_tokenizer(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Tokenizer file not found: {path.resolve()}"
        )

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    # Restore ordered BPE merge rules.
    merges = {
        (first_id, second_id): new_id
        for first_id, second_id, new_id in data["merges"]
    }

    vocab = build_vocab(merges)

    return merges, vocab, data["vocab_size"]

In [10]:
tokenizer_path = Path("tokenizer/tokenizer/tokenizer.json")
if not tokenizer_path.exists():
    tokenizer_path = Path("..") / tokenizer_path

merges, vocab, vocab_size = load_tokenizer(tokenizer_path)

print("Tokenizer loaded")
print("Vocabulary size:", vocab_size)
print("Number of merges:", len(merges))

Tokenizer loaded
Vocabulary size: 1000
Number of merges: 744


In [11]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


def build_fast_tokenizer(merges, vocab):
    byte_values = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )

    unicode_values = byte_values.copy()
    extra_index = 0

    for byte_value in range(256):
        if byte_value not in byte_values:
            byte_values.append(byte_value)
            unicode_values.append(256 + extra_index)
            extra_index += 1

    byte_encoder = {
        byte_value: chr(unicode_value)
        for byte_value, unicode_value
        in zip(byte_values, unicode_values)
    }

    def bytes_to_token_string(byte_sequence):
        return "".join(
            byte_encoder[byte_value]
            for byte_value in byte_sequence
        )

    fast_vocab = {
        bytes_to_token_string(byte_sequence): token_id
        for token_id, byte_sequence in vocab.items()
    }

    fast_merges = [
        (
            bytes_to_token_string(vocab[first_id]),
            bytes_to_token_string(vocab[second_id]),
        )
        for first_id, second_id in merges
    ]

    tokenizer = Tokenizer(
        BPE(
            vocab=fast_vocab,
            merges=fast_merges,
        )
    )

    tokenizer.pre_tokenizer = ByteLevel(
        add_prefix_space=False,
        use_regex=False,
    )

    tokenizer.decoder = ByteLevelDecoder()

    return tokenizer


fast_tokenizer = build_fast_tokenizer(
    merges=merges,
    vocab=vocab,
)

print("Fast tokenizer created")
print("Vocabulary size:", fast_tokenizer.get_vocab_size())

Fast tokenizer created
Vocabulary size: 1000


In [14]:
def format_conversation(conversation):
    text = ""

    for message in conversation["messages"]:
        role = message["role"]
        content = message["content"]

        text += f"{role}: {content}\n"

    return text

In [19]:
all_input_ids = []
all_text = ""
for conversation in data:
    text = format_conversation(conversation)
    all_text += text + "<newLINE>"+  "\n" 

    encoding = fast_tokenizer.encode(text)

    all_input_ids.append(encoding.ids)

In [22]:
print("Conversations:", len(data))
print("Encoded conversations:", len(all_input_ids))

Conversations: 620
Encoded conversations: 620


In [24]:
lengths = [len(ids) for ids in all_input_ids]

print("Total conversations:", len(all_input_ids))
print("Minimum tokens:", min(lengths))
print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))

Total conversations: 620
Minimum tokens: 164
Maximum tokens: 3208
Average tokens: 1285.116129032258


In [25]:
import numpy as np

print("P50:", np.percentile(lengths, 50))
print("P75:", np.percentile(lengths, 75))
print("P90:", np.percentile(lengths, 90))
print("P95:", np.percentile(lengths, 95))
print("P99:", np.percentile(lengths, 99))

P50: 1247.0
P75: 1526.25
P90: 1823.3000000000002
P95: 2068.1
P99: 2479.43


In [26]:
import re

CONTEXT_LENGTH = 512


def token_length(text):
    """Return number of tokens using your existing tokenizer."""
    return len(fast_tokenizer.encode(text).ids)


def format_pair(user_text, assistant_text):
    """
    Format one user-assistant pair exactly as the model will see it.
    """
    return (
        f"user: {user_text}\n"
        f"assistant: {assistant_text}"
    )


def split_into_sentences(text):
    """
    Split text at paragraph and sentence boundaries.

    Keeps sentence punctuation when possible.
    """
    text = text.strip()

    # First normalize excessive whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Split on:
    # 1. paragraph boundaries
    # 2. sentence endings followed by whitespace
    parts = re.split(
        r'\n\s*\n|(?<=[.!?])\s+',
        text
    )

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]


def hard_split_text(text, max_tokens):
    """
    Fallback for a single sentence that itself is too long.

    Splits using words while respecting max_tokens.
    """

    words = text.split()

    chunks = []
    current_words = []

    for word in words:

        candidate_words = current_words + [word]
        candidate = " ".join(candidate_words)

        if token_length(candidate) <= max_tokens:
            current_words.append(word)

        else:
            if current_words:
                chunks.append(" ".join(current_words))

            current_words = [word]

            # Extremely unusual case:
            # one word itself exceeds the token budget
            if token_length(word) > max_tokens:

                word_ids = fast_tokenizer.encode(word).ids

                for i in range(0, len(word_ids), max_tokens):
                    ids_chunk = word_ids[i:i + max_tokens]

                    chunks.append(
                        fast_tokenizer.decode(ids_chunk)
                    )

                current_words = []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def split_assistant_response(
    user_text,
    assistant_text,
    context_length=512
):
    """
    Keep the user question in every sample and split
    assistant response so every formatted sample <= context_length.
    """

    # Number of tokens used before assistant answer
    prefix = (
        f"user: {user_text}\n"
        f"assistant: "
    )

    prefix_tokens = token_length(prefix)

    available_tokens = context_length - prefix_tokens

    if available_tokens <= 0:
        return []

    sentences = split_into_sentences(assistant_text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:

        if current_chunk:
            candidate = current_chunk + " " + sentence
        else:
            candidate = sentence

        full_candidate = prefix + candidate

        # Sentence fits into current chunk
        if token_length(full_candidate) <= context_length:
            current_chunk = candidate

        else:
            # Save existing chunk first
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            # Check whether this sentence can fit by itself
            full_sentence = prefix + sentence

            if token_length(full_sentence) <= context_length:
                current_chunk = sentence

            else:
                # Sentence itself is too long.
                # Fall back to word-level splitting.
                smaller_chunks = hard_split_text(
                    sentence,
                    available_tokens
                )

                chunks.extend(smaller_chunks)

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


def create_training_samples(
    cleaned_data,
    context_length=512
):
    """
    Convert cleaned conversations into <=512-token
    user-assistant SFT samples.
    """

    training_samples = []

    stats = {
        "original_conversations": len(cleaned_data),
        "pairs_found": 0,
        "pairs_that_fit": 0,
        "pairs_split": 0,
        "samples_created": 0,
        "skipped": 0,
    }

    for conversation in cleaned_data:

        messages = conversation.get("messages", [])

        i = 0

        while i < len(messages) - 1:

            current = messages[i]
            next_message = messages[i + 1]

            # Look for user -> assistant pair
            if (
                current.get("role") == "user"
                and next_message.get("role") == "assistant"
            ):

                user_text = current["content"].strip()
                assistant_text = next_message["content"].strip()

                stats["pairs_found"] += 1

                if not user_text or not assistant_text:
                    stats["skipped"] += 1
                    i += 2
                    continue

                full_text = format_pair(
                    user_text,
                    assistant_text
                )

                if token_length(full_text) <= context_length:

                    encoding = fast_tokenizer.encode(full_text)

                    training_samples.append({
                        "text": full_text,
                        "input_ids": encoding.ids,
                        "user": user_text,
                        "assistant": assistant_text,
                        "was_split": False
                    })

                    stats["pairs_that_fit"] += 1

                else:

                    assistant_chunks = split_assistant_response(
                        user_text,
                        assistant_text,
                        context_length
                    )

                    if not assistant_chunks:
                        stats["skipped"] += 1
                        i += 2
                        continue

                    stats["pairs_split"] += 1

                    for chunk in assistant_chunks:

                        text = format_pair(
                            user_text,
                            chunk
                        )

                        encoding = fast_tokenizer.encode(text)

                        # Final safety check
                        if len(encoding.ids) <= context_length:

                            training_samples.append({
                                "text": text,
                                "input_ids": encoding.ids,
                                "user": user_text,
                                "assistant": chunk,
                                "was_split": True
                            })

                        else:
                            print(
                                "WARNING: Sample still >",
                                context_length,
                                "tokens:",
                                len(encoding.ids)
                            )

                i += 2

            else:
                i += 1

    stats["samples_created"] = len(training_samples)

    return training_samples, stats

In [ ]:
training_samples, stats = create_training_samples(
    data,
    context_length=512
)